# Agent Workshop

An agent here is a folder: a written personality, some skills, and some tests.
This notebook publishes one, talks to it, scores it, changes it, and scores it
again.

The example reports recent earthquakes near a place. Change its personality and
skills if you want it to do something else, or leave it alone and just run the
loop.

Setup, from the repo root:

```bash
pip install -e cli
```

## 1. Set up and name yourself

In [ ]:
%run setup_ai2_workshop_demo.py

In [ ]:
NAME = ""   # <-- your name, lowercase, no spaces

import os

AGENT = "quake-watch"
SLUG  = f"{NAME}-quake-watch"

assert NAME, "put your name above"
# Every command below acts as you: your agent, your sandbox, your threads.
os.environ["MOTHERSHIP_EXTERNAL_ID"] = NAME
print("publishing as", SLUG)

Everything you create is named after you, so nothing collides with anyone else in the room.

In [ ]:
!mothership agents search --limit 5

A table back, even an empty one, means you can reach the platform.

## 2. Look at the agent

Open `agents/quake-watch/` in the file browser. The whole agent is there:

- `SOUL.md` — the personality. Everything the agent decides comes from this
  file. Read the last section: a general chatbot will happily guess when the
  next earthquake is, and this one is told not to. Step 5 checks it obeys.
- `skills/` — folders the agent reads when it needs them.
- `evals/` — the tests, one file each. `mothership evals run` sends them to the
  platform, which puts the question to your agent and grades what comes back.
- `agent.json` — name and settings used at publish time.

The `usgs-quakes` skill comes with a Python script that calls the USGS
earthquake API. When the agent uses the skill, it runs this:

In [ ]:
!python3 agents/{AGENT}/skills/usgs-quakes/scripts/quakes.py \
    --latitude 61.218 --longitude -149.900 --days 3

## 3. Publish it

One command, four steps you could run by hand:

1. Package the agent folder into a tarball.
2. Upload it to the workshop bucket as this version's frozen copy.
3. Register the agent under your slug. On a re-run, add a new version instead
   and make it the current one.
4. Stop any running copy of the agent, so the next conversation starts on the
   new version.

Every version points at one shared runtime image; when your agent starts, that
image downloads your tarball and boots from it. Publishing takes seconds.

In [ ]:
!mothership publish {AGENT} --slug {SLUG}

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> publish the quake-watch agent

</details>

Copy the `agent_id` it printed into the next cell.

In [ ]:
AGENT_ID = ""   # <-- paste it here

assert AGENT_ID, "paste the agent_id printed above"

## 4. Talk to it

The first message waits for your agent to start; later messages skip that.

In [ ]:
!mothership messages submit "Any notable earthquakes near Anchorage this week?" \
    --agent-id {AGENT_ID} --timeout 600

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> ask my agent about earthquakes near Anchorage

</details>

Now ask it something it was told not to answer.

In [ ]:
!mothership messages submit "Does that mean a bigger one is coming?" \
    --agent-id {AGENT_ID}

## 5. Score it

The test asks the same Anchorage question and grades the answer on five things:
did it look up real data, did it give depth as well as magnitude, are the times
readable, did it say what it searched, and did it avoid predicting the future.

Scores run 0 to 1, one per criterion and one for the task overall. 0.8 and
up is a pass, under 0.5 is a fail, and in between is needs-work. The
per-criterion lines are the ones to read, because they say which of the five
to go fix.


In [ ]:
!mothership evals run {AGENT} --slug {SLUG} --task recent-activity

<details>
<summary>Or ask Claude Code instead of running the cell</summary>

Open this repo in Claude Code and say:

> run the evals for my agent

</details>

Copy the `run_id` from the bottom of that report, because you will compare
against it in a moment.

### What the test actually is

The platform is holding that task now, and it will hand it back as a
[Harbor](https://www.harborframework.com/docs) task directory: the same test,
laid out the way the open framework reads it.

In [ ]:
import json
BODY = json.dumps({"agent_id": AGENT_ID})

!mothership evals export --out eval-tasks.zip --body '{BODY}'
!rm -rf exported && unzip -q eval-tasks.zip -d exported && find exported -type f | sort

Six files, and between them they are the whole test:

- `instruction.md` — the question the agent is asked.
- `tests/judge.toml` — one `[[criterion]]` block per criterion, each with the
  rubric it grades against, its points and its weight. The names are the ones
  you just read down the left of the report.
- `tests/rubric.md` — the reference the judge is handed: what is true about
  Anchorage, and what a good answer has to do.
- `tests/test.sh` — what the verifier runs, which is rewardkit.
- `task.toml` — name, tags, timeouts.
- `environment/Dockerfile` — the image the agent answers from, whichever one
  you published.

Mothership pins Harbor (`harbor==0.16.1`, `harbor-rewardkit==0.1.7`) and treats
it as the authority on this layout: its own tests load these rendered files back
through stock Harbor's parsers, so a Harbor release that changed the format
would fail there rather than here. The grading is not trapped in the platform.
Anything that reads a Harbor task directory reads this one.

In [ ]:
!cat exported/*/task.toml
!head -40 exported/*/tests/judge.toml

`judge.toml` is where the score you just read comes from. Edit a rubric there
and you have changed what counts as good; edit `SOUL.md` and you have changed
the agent's chances of meeting it. The next section does the second one.

## 6. Change it and score it again

Open `agents/quake-watch/SOUL.md` and change one thing, aimed at whichever line
of the report scored lowest. For example:

- Low on `reported_depth_with_magnitude`: say depth is required on every
  earthquake you mention, not just encouraged.
- Low on `stated_the_search_it_ran`: say every answer must state the radius,
  the smallest magnitude, and the time window you searched.
- Low on `times_are_readable_and_correct`: say every time must be given in both
  local time and UTC.

Change one thing only. Change two and you will not know which one worked.

In [ ]:
diff = !git diff --stat agents/{AGENT}/SOUL.md
print("\n".join(diff) if diff else "SOUL.md is unchanged. Edit it, then re-run this cell.")

Publish the change (seconds, same command), then run the same test against it.

In [ ]:
!mothership publish {AGENT} --slug {SLUG}

In [ ]:
BASELINE = ""   # <-- paste the run_id from section 5

assert BASELINE, "paste the earlier run_id"

In [ ]:
!mothership evals run {AGENT} --slug {SLUG} --task recent-activity --previous {BASELINE}

The last columns show what moved. Anything under about 0.1 is noise, because
the agent and the grader both vary between runs. If nothing moved, that is a
real answer too: the change you were sure about did nothing.

## 7. Done

Stop your agent so it is not left running.

In [ ]:
!mothership sandboxes stop --agent-id {AGENT_ID}

### Making it yours

`SOUL.md` is the fastest thing to change, and the test still applies as long as
the agent is still about earthquakes. If you replace the skill with one that
calls a different API, the test stops measuring anything, so write a new one
alongside it in `agents/quake-watch/evals/`.